# Notebook 13: Animated Visualizations

**One Sensor, One Year — India Breathes**

The data should *move*. These animations show India's grid breathing through 2024 — consumption rising and falling, emissions accumulating, fuel sources racing against each other, seasons painting themselves across the calendar.

All animations use Plotly with play/pause controls. Hit ▶ to watch.

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

df = pd.read_csv('../data/processed/india_2024_derived.csv', parse_dates=['date'])

fuel_cols = ['coal', 'lignite', 'gas', 'nuclear', 'hydro', 'wind', 'solar', 'other_re']
for col in fuel_cols:
    df[col] = df[col].fillna(0)

df['wind_solar'] = df['wind'] + df['solar']
df['month'] = df['date'].dt.month
df['week'] = df['date'].dt.isocalendar().week.astype(int)
df['dow'] = df['date'].dt.dayofweek
df['day_of_year'] = df['date'].dt.dayofyear

# Palettes
earth_sky = {
    'coal': '#3D2B1F', 'lignite': '#6B4226', 'gas': '#4A90D9',
    'nuclear': '#2EC4B6', 'hydro': '#1B4F72', 'wind': '#85C1E9',
    'solar': '#F5B041', 'other_re': '#A569BD',
}
bg = '#FAFAF5'
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

print(f'Data ready: {len(df)} days')

---
## 1. Racing Lines — Wind+Solar vs Gas vs Nuclear

Watch the cumulative generation of wind+solar pull away from gas and nuclear through the year. The gap *accelerates* during monsoon.

In [ ]:
# Cumulative generation for key sources
df['cum_wind_solar'] = df['wind_solar'].cumsum()
df['cum_gas'] = df['gas'].cumsum()
df['cum_nuclear'] = df['nuclear'].cumsum()
df['cum_hydro'] = df['hydro'].cumsum()
df['cum_solar'] = df['solar'].cumsum()
df['cum_wind'] = df['wind'].cumsum()

# Build frames — one per week (52 frames for smooth animation)
frames = []
# Use every 3rd day for smoother animation (122 frames)
step = 3
frame_indices = list(range(0, len(df), step)) + [len(df)-1]

for idx in frame_indices:
    d = df.iloc[:idx+1]
    frame = go.Frame(
        data=[
            go.Scatter(x=d['date'], y=d['cum_wind_solar']/1e3, mode='lines',
                      line=dict(width=3, color='#E67E22'), name='Wind+Solar'),
            go.Scatter(x=d['date'], y=d['cum_gas']/1e3, mode='lines',
                      line=dict(width=3, color=earth_sky['gas']), name='Gas'),
            go.Scatter(x=d['date'], y=d['cum_nuclear']/1e3, mode='lines',
                      line=dict(width=3, color=earth_sky['nuclear']), name='Nuclear'),
            go.Scatter(x=d['date'], y=d['cum_hydro']/1e3, mode='lines',
                      line=dict(width=3, color=earth_sky['hydro']), name='Hydro'),
        ],
        name=str(df.iloc[idx]['date'].strftime('%b %d')),
    )
    frames.append(frame)

# Initial state (first frame)
d0 = df.iloc[:1]
fig = go.Figure(
    data=[
        go.Scatter(x=d0['date'], y=d0['cum_wind_solar']/1e3, mode='lines',
                  line=dict(width=3, color='#E67E22'), name='Wind+Solar'),
        go.Scatter(x=d0['date'], y=d0['cum_gas']/1e3, mode='lines',
                  line=dict(width=3, color=earth_sky['gas']), name='Gas'),
        go.Scatter(x=d0['date'], y=d0['cum_nuclear']/1e3, mode='lines',
                  line=dict(width=3, color=earth_sky['nuclear']), name='Nuclear'),
        go.Scatter(x=d0['date'], y=d0['cum_hydro']/1e3, mode='lines',
                  line=dict(width=3, color=earth_sky['hydro']), name='Hydro'),
    ],
    frames=frames,
)

fig.update_layout(
    title=dict(text='Racing Lines — Cumulative Generation by Source (TWh)', font=dict(size=18)),
    xaxis=dict(range=[df['date'].min(), df['date'].max()]),
    yaxis=dict(range=[0, df['cum_wind_solar'].max()/1e3 * 1.1], title='Cumulative TWh'),
    plot_bgcolor=bg, paper_bgcolor=bg,
    height=550,
    legend=dict(orientation='h', y=-0.1, x=0.5, xanchor='center'),
    updatemenus=[dict(
        type='buttons', showactive=False,
        y=1.15, x=0.5, xanchor='center',
        buttons=[
            dict(label='▶ Play', method='animate',
                 args=[None, dict(frame=dict(duration=50, redraw=True),
                                  fromcurrent=True, mode='immediate')]),
            dict(label='⏸ Pause', method='animate',
                 args=[[None], dict(frame=dict(duration=0, redraw=False),
                                    mode='immediate')]),
        ]
    )],
    sliders=[dict(
        active=0, pad=dict(t=50),
        steps=[dict(args=[[f.name], dict(frame=dict(duration=0, redraw=True), mode='immediate')],
                    method='animate', label=f.name) for f in frames],
        currentvalue=dict(prefix='Date: ', font=dict(size=14)),
    )],
)

fig.show()
print('Wind+Solar pulls away from Gas and Nuclear immediately.')
print(f'By year end: W+S = {df["cum_wind_solar"].iloc[-1]/1e3:.0f} TWh, Gas = {df["cum_gas"].iloc[-1]/1e3:.0f} TWh, Nuclear = {df["cum_nuclear"].iloc[-1]/1e3:.0f} TWh')
print(f'Wind+Solar produced {df["cum_wind_solar"].iloc[-1]/df["cum_gas"].iloc[-1]:.1f}x more than Gas')

---
## 2. Monthly Fuel Mix — Animated Donut

Watch the fuel mix morph month by month. Coal dominates in winter, then hydro and wind swell during monsoon, squeezing coal's share.

In [ ]:
# Monthly averages for each fuel
fuel_labels = ['Coal', 'Lignite', 'Gas', 'Nuclear', 'Hydro', 'Wind', 'Solar', 'Other RE']
fuel_colors = [earth_sky[f] for f in fuel_cols]

monthly_totals = df.groupby('month')[fuel_cols].sum()

frames = []
for m in range(1, 13):
    vals = monthly_totals.loc[m].values
    total = vals.sum()
    frame = go.Frame(
        data=[go.Pie(
            labels=fuel_labels, values=vals,
            marker=dict(colors=fuel_colors),
            hole=0.45,
            textinfo='label+percent',
            textposition='inside',
            hovertemplate='%{label}: %{value:.0f} MU (%{percent})<extra></extra>',
            sort=False,
        )],
        name=month_names[m-1],
        layout=dict(title=dict(
            text=f'Monthly Fuel Mix — {month_names[m-1]} 2024<br><span style="font-size:14px">Total: {total/1e3:.1f} TWh</span>',
            font=dict(size=18),
        )),
    )
    frames.append(frame)

# Start with January
vals0 = monthly_totals.loc[1].values
total0 = vals0.sum()

fig = go.Figure(
    data=[go.Pie(
        labels=fuel_labels, values=vals0,
        marker=dict(colors=fuel_colors),
        hole=0.45,
        textinfo='label+percent',
        textposition='inside',
        hovertemplate='%{label}: %{value:.0f} MU (%{percent})<extra></extra>',
        sort=False,
    )],
    frames=frames,
)

fig.update_layout(
    title=dict(text=f'Monthly Fuel Mix — Jan 2024<br><span style="font-size:14px">Total: {total0/1e3:.1f} TWh</span>',
              font=dict(size=18)),
    plot_bgcolor=bg, paper_bgcolor=bg,
    height=550, width=600,
    updatemenus=[dict(
        type='buttons', showactive=False,
        y=1.12, x=0.5, xanchor='center',
        buttons=[
            dict(label='▶ Play', method='animate',
                 args=[None, dict(frame=dict(duration=800, redraw=True),
                                  fromcurrent=True, mode='immediate',
                                  transition=dict(duration=400))]),
            dict(label='⏸ Pause', method='animate',
                 args=[[None], dict(frame=dict(duration=0, redraw=False),
                                    mode='immediate')]),
        ]
    )],
    sliders=[dict(
        active=0, pad=dict(t=60),
        steps=[dict(args=[[f.name], dict(frame=dict(duration=0, redraw=True), mode='immediate',
                                          transition=dict(duration=300))],
                    method='animate', label=f.name) for f in frames],
        currentvalue=dict(prefix='Month: ', font=dict(size=14)),
    )],
)

fig.show()
print('Watch coal\'s share shrink during monsoon (Jul-Sep) as hydro and wind surge.')

---
## 3. Weekly Heatmap — Painting the Year

Watch the heatmap fill in week by week. See winter cool tones give way to summer heat, then the monsoon pattern, and back to winter. The Sunday column stays cooler throughout.

In [ ]:
dow_order = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

# Build the full heatmap matrix
pivot = df.pivot_table(values='total', index='week', columns='dow', aggfunc='mean')
pivot.columns = dow_order
all_weeks = sorted(pivot.index)

z_max = pivot.values[~np.isnan(pivot.values)].max()
z_min = pivot.values[~np.isnan(pivot.values)].min()

# Build frames — reveal one week at a time
frames = []
for w_idx in range(1, len(all_weeks) + 1):
    # Show weeks revealed so far, mask the rest with NaN
    z_partial = np.full_like(pivot.values, np.nan, dtype=float)
    z_partial[:w_idx, :] = pivot.values[:w_idx, :]
    
    # Figure out the approximate month
    last_week = all_weeks[w_idx - 1]
    approx_day = (last_week - 1) * 7 + 1
    approx_month = min(12, max(1, (approx_day - 1) // 30 + 1))
    
    frame = go.Frame(
        data=[go.Heatmap(
            z=z_partial,
            x=dow_order,
            y=[f'W{w}' for w in all_weeks],
            colorscale='YlOrRd',
            zmin=z_min, zmax=z_max,
            hovertemplate='Week %{y}, %{x}: %{z:.0f} MU<extra></extra>',
            colorbar_title='MU/day',
        )],
        name=f'W{all_weeks[w_idx-1]} ({month_names[approx_month-1]})',
    )
    frames.append(frame)

# Start empty
z_init = np.full_like(pivot.values, np.nan, dtype=float)
z_init[0, :] = pivot.values[0, :]

fig = go.Figure(
    data=[go.Heatmap(
        z=z_init,
        x=dow_order,
        y=[f'W{w}' for w in all_weeks],
        colorscale='YlOrRd',
        zmin=z_min, zmax=z_max,
        hovertemplate='Week %{y}, %{x}: %{z:.0f} MU<extra></extra>',
        colorbar_title='MU/day',
    )],
    frames=frames,
)

fig.update_layout(
    title=dict(text='Weekly Heatmap — Total Generation Painting Itself', font=dict(size=18)),
    xaxis_title='Day of Week',
    yaxis=dict(autorange='reversed', title='Week of Year'),
    plot_bgcolor=bg, paper_bgcolor=bg,
    height=800, width=550,
    updatemenus=[dict(
        type='buttons', showactive=False,
        y=1.05, x=0.5, xanchor='center',
        buttons=[
            dict(label='▶ Play', method='animate',
                 args=[None, dict(frame=dict(duration=150, redraw=True),
                                  fromcurrent=True, mode='immediate')]),
            dict(label='⏸ Pause', method='animate',
                 args=[[None], dict(frame=dict(duration=0, redraw=False),
                                    mode='immediate')]),
        ]
    )],
    sliders=[dict(
        active=0, pad=dict(t=50),
        steps=[dict(args=[[f.name], dict(frame=dict(duration=0, redraw=True), mode='immediate')],
                    method='animate', label=f.name) for f in frames],
        currentvalue=dict(prefix='', font=dict(size=12)),
    )],
)

fig.show()
print('Watch summer weeks glow hot (dark red), monsoon cool slightly, winter returns.')
print('The Sunday column is consistently lighter — the weekend dip is visible all year.')

---
## 4. Emissions Heatmap — CO2 Intensity Through the Year

Same weekly heatmap format, but colored by emissions intensity (tCO2/GWh). Dirtiest days glow dark. Watch the monsoon bring relief.

In [ ]:
# Emissions intensity heatmap
pivot_ei = df.pivot_table(values='emissions_intensity', index='week', columns='dow', aggfunc='mean')
pivot_ei.columns = dow_order

ei_max = pivot_ei.values[~np.isnan(pivot_ei.values)].max()
ei_min = pivot_ei.values[~np.isnan(pivot_ei.values)].min()

# Build frames
frames_ei = []
for w_idx in range(1, len(all_weeks) + 1):
    z_partial = np.full_like(pivot_ei.values, np.nan, dtype=float)
    z_partial[:w_idx, :] = pivot_ei.values[:w_idx, :]
    
    last_week = all_weeks[w_idx - 1]
    approx_day = (last_week - 1) * 7 + 1
    approx_month = min(12, max(1, (approx_day - 1) // 30 + 1))
    
    frame = go.Frame(
        data=[go.Heatmap(
            z=z_partial,
            x=dow_order,
            y=[f'W{w}' for w in all_weeks],
            colorscale='RdYlGn_r',  # red = dirty, green = clean
            zmin=ei_min, zmax=ei_max,
            hovertemplate='Week %{y}, %{x}: %{z:.0f} tCO2/GWh<extra></extra>',
            colorbar_title='tCO2/GWh',
        )],
        name=f'W{all_weeks[w_idx-1]} ({month_names[approx_month-1]})',
    )
    frames_ei.append(frame)

z_init_ei = np.full_like(pivot_ei.values, np.nan, dtype=float)
z_init_ei[0, :] = pivot_ei.values[0, :]

fig = go.Figure(
    data=[go.Heatmap(
        z=z_init_ei,
        x=dow_order,
        y=[f'W{w}' for w in all_weeks],
        colorscale='RdYlGn_r',
        zmin=ei_min, zmax=ei_max,
        hovertemplate='Week %{y}, %{x}: %{z:.0f} tCO2/GWh<extra></extra>',
        colorbar_title='tCO2/GWh',
    )],
    frames=frames_ei,
)

fig.update_layout(
    title=dict(text='Emissions Intensity Heatmap — How Dirty Is Each Day?', font=dict(size=18)),
    xaxis_title='Day of Week',
    yaxis=dict(autorange='reversed', title='Week of Year'),
    plot_bgcolor=bg, paper_bgcolor=bg,
    height=800, width=550,
    updatemenus=[dict(
        type='buttons', showactive=False,
        y=1.05, x=0.5, xanchor='center',
        buttons=[
            dict(label='▶ Play', method='animate',
                 args=[None, dict(frame=dict(duration=150, redraw=True),
                                  fromcurrent=True, mode='immediate')]),
            dict(label='⏸ Pause', method='animate',
                 args=[[None], dict(frame=dict(duration=0, redraw=False),
                                    mode='immediate')]),
        ]
    )],
    sliders=[dict(
        active=0, pad=dict(t=50),
        steps=[dict(args=[[f.name], dict(frame=dict(duration=0, redraw=True), mode='immediate')],
                    method='animate', label=f.name) for f in frames_ei],
        currentvalue=dict(prefix='', font=dict(size=12)),
    )],
)

fig.show()
print('Red = dirty (high coal share). Green = cleaner (monsoon months).')
print('Side by side with the generation heatmap: summer is HOT + DIRTY, monsoon is HOT + CLEANER.')

---
## 5. The CO2 Race — Actual vs Counterfactual

Watch two lines race: actual CO2 emissions vs what they'd be if all clean energy were replaced by coal. The gap is what renewables save.

In [ ]:
# Compute counterfactual
coal_ef = df['co2_coal'].sum() / df['coal'].sum()  # kt per GWh
clean_cols = ['nuclear', 'hydro', 'wind', 'solar', 'other_re']
df['clean_gen'] = df[clean_cols].sum(axis=1)
df['emissions_avoided'] = df['clean_gen'] * coal_ef
df['co2_counterfactual'] = df['co2_total'] + df['emissions_avoided']
df['cum_co2_actual'] = df['co2_total'].cumsum() / 1e3  # Mt
df['cum_co2_cf'] = df['co2_counterfactual'].cumsum() / 1e3

# Milestone dates
milestones = [250, 500, 750, 1000, 1250]

# Build frames
frames = []
step = 3
frame_indices = list(range(0, len(df), step)) + [len(df)-1]

for idx in frame_indices:
    d = df.iloc[:idx+1]
    frame = go.Frame(
        data=[
            go.Scatter(x=d['date'], y=d['cum_co2_cf'],
                      fill='tozeroy', fillcolor='rgba(192,57,43,0.1)',
                      line=dict(width=2.5, color='#C0392B', dash='dash'),
                      name='Without Clean Energy'),
            go.Scatter(x=d['date'], y=d['cum_co2_actual'],
                      fill='tozeroy', fillcolor='rgba(46,196,182,0.2)',
                      line=dict(width=2.5, color='#2EC4B6'),
                      name='Actual Emissions'),
        ],
        name=str(df.iloc[idx]['date'].strftime('%b %d')),
    )
    frames.append(frame)

d0 = df.iloc[:1]
fig = go.Figure(
    data=[
        go.Scatter(x=d0['date'], y=d0['cum_co2_cf'],
                  fill='tozeroy', fillcolor='rgba(192,57,43,0.1)',
                  line=dict(width=2.5, color='#C0392B', dash='dash'),
                  name='Without Clean Energy'),
        go.Scatter(x=d0['date'], y=d0['cum_co2_actual'],
                  fill='tozeroy', fillcolor='rgba(46,196,182,0.2)',
                  line=dict(width=2.5, color='#2EC4B6'),
                  name='Actual Emissions'),
    ],
    frames=frames,
)

# Add milestone lines
for ms in milestones:
    fig.add_hline(y=ms, line_dash='dot', line_color='#DDD', line_width=1,
                  annotation_text=f'{ms} Mt', annotation_position='top left',
                  annotation_font_color='#AAA')

fig.update_layout(
    title=dict(text='The CO2 Race — Actual vs "No Clean Energy" Counterfactual', font=dict(size=18)),
    xaxis=dict(range=[df['date'].min(), df['date'].max()]),
    yaxis=dict(range=[0, df['cum_co2_cf'].max() * 1.05], title='Cumulative CO2 (Mt)'),
    plot_bgcolor=bg, paper_bgcolor=bg,
    height=550,
    legend=dict(orientation='h', y=-0.1, x=0.5, xanchor='center'),
    updatemenus=[dict(
        type='buttons', showactive=False,
        y=1.15, x=0.5, xanchor='center',
        buttons=[
            dict(label='▶ Play', method='animate',
                 args=[None, dict(frame=dict(duration=50, redraw=True),
                                  fromcurrent=True, mode='immediate')]),
            dict(label='⏸ Pause', method='animate',
                 args=[[None], dict(frame=dict(duration=0, redraw=False),
                                    mode='immediate')]),
        ]
    )],
    sliders=[dict(
        active=0, pad=dict(t=50),
        steps=[dict(args=[[f.name], dict(frame=dict(duration=0, redraw=True), mode='immediate')],
                    method='animate', label=f.name) for f in frames],
        currentvalue=dict(prefix='Date: ', font=dict(size=14)),
    )],
)

fig.show()
gap = df['cum_co2_cf'].iloc[-1] - df['cum_co2_actual'].iloc[-1]
print(f'The gap at year end: {gap:.0f} Mt CO2 — that\'s what clean energy saved.')
print(f'Actual: {df["cum_co2_actual"].iloc[-1]:.0f} Mt | Without clean: {df["cum_co2_cf"].iloc[-1]:.0f} Mt')

---
## 6. Animated Stacked Area — The Breathing Grid

Watch the full generation stack build day by day. Fossil fuels below, clean above the midline. The grid breathes — clean side swells during monsoon, shrinks in winter.

In [ ]:
# Mirrored heartbeat animation — fossil down, clean up
fossil_fuels = ['coal', 'lignite', 'gas']
clean_fuels = ['nuclear', 'hydro', 'wind', 'solar', 'other_re']

fossil_labels = ['Coal', 'Lignite', 'Gas']
clean_labels = ['Nuclear', 'Hydro', 'Wind', 'Solar', 'Other RE']

fossil_colors = ['#922B21', '#C0392B', '#E74C3C']
clean_colors = ['#2EC4B6', '#1B4F72', '#85C1E9', '#F5B041', '#A569BD']

# Pre-compute stacked values
fossil_stack = np.zeros((len(df), len(fossil_fuels) + 1))
for i, fuel in enumerate(fossil_fuels):
    fossil_stack[:, i+1] = fossil_stack[:, i] + df[fuel].values

clean_stack = np.zeros((len(df), len(clean_fuels) + 1))
for i, fuel in enumerate(clean_fuels):
    clean_stack[:, i+1] = clean_stack[:, i] + df[fuel].values

# Build frames — reveal 7 days at a time
frames = []
step = 7
frame_indices = list(range(step, len(df), step)) + [len(df)-1]

for idx in frame_indices:
    traces = []
    dates = df['date'].iloc[:idx+1]
    
    # Fossil (negative)
    for i, (fuel, label, color) in enumerate(zip(fossil_fuels, fossil_labels, fossil_colors)):
        traces.append(go.Scatter(
            x=dates, y=-fossil_stack[:idx+1, i+1],
            fill='tonexty' if i > 0 else 'tozeroy',
            fillcolor=color, line=dict(width=0, color=color),
            name=label, stackgroup=None,
            hovertemplate=f'{label}: %{{text:.0f}} MU<extra></extra>',
            text=df[fuel].iloc[:idx+1],
        ))
    
    # Clean (positive)
    for i, (fuel, label, color) in enumerate(zip(clean_fuels, clean_labels, clean_colors)):
        traces.append(go.Scatter(
            x=dates, y=clean_stack[:idx+1, i+1],
            fill='tonexty' if i > 0 else 'tozeroy',
            fillcolor=color, line=dict(width=0, color=color),
            name=label,
            hovertemplate=f'{label}: %{{text:.0f}} MU<extra></extra>',
            text=df[fuel].iloc[:idx+1],
        ))
    
    frame = go.Frame(data=traces, name=df.iloc[idx]['date'].strftime('%b %d'))
    frames.append(frame)

# Initial state
init_idx = step
init_traces = []
dates0 = df['date'].iloc[:init_idx+1]

for i, (fuel, label, color) in enumerate(zip(fossil_fuels, fossil_labels, fossil_colors)):
    init_traces.append(go.Scatter(
        x=dates0, y=-fossil_stack[:init_idx+1, i+1],
        fill='tonexty' if i > 0 else 'tozeroy',
        fillcolor=color, line=dict(width=0, color=color),
        name=label,
    ))

for i, (fuel, label, color) in enumerate(zip(clean_fuels, clean_labels, clean_colors)):
    init_traces.append(go.Scatter(
        x=dates0, y=clean_stack[:init_idx+1, i+1],
        fill='tonexty' if i > 0 else 'tozeroy',
        fillcolor=color, line=dict(width=0, color=color),
        name=label,
    ))

fig = go.Figure(data=init_traces, frames=frames)

max_fossil = fossil_stack[:, -1].max()
max_clean = clean_stack[:, -1].max()
y_range = max(max_fossil, max_clean) * 1.1

fig.add_hline(y=0, line_color='#333', line_width=1.5)

fig.update_layout(
    title=dict(text='The Breathing Grid — Fossil ↓ Clean ↑', font=dict(size=18)),
    xaxis=dict(range=[df['date'].min(), df['date'].max()]),
    yaxis=dict(range=[-y_range, y_range], title='Generation (MU/day)',
              zeroline=True, zerolinewidth=2, zerolinecolor='#333'),
    plot_bgcolor=bg, paper_bgcolor=bg,
    height=550,
    legend=dict(orientation='h', y=-0.12, x=0.5, xanchor='center'),
    updatemenus=[dict(
        type='buttons', showactive=False,
        y=1.15, x=0.5, xanchor='center',
        buttons=[
            dict(label='▶ Play', method='animate',
                 args=[None, dict(frame=dict(duration=80, redraw=True),
                                  fromcurrent=True, mode='immediate')]),
            dict(label='⏸ Pause', method='animate',
                 args=[[None], dict(frame=dict(duration=0, redraw=False),
                                    mode='immediate')]),
        ]
    )],
    sliders=[dict(
        active=0, pad=dict(t=50),
        steps=[dict(args=[[f.name], dict(frame=dict(duration=0, redraw=True), mode='immediate')],
                    method='animate', label=f.name) for f in frames],
        currentvalue=dict(prefix='Date: ', font=dict(size=14)),
    )],
)

fig.show()
print('Below zero: fossil (coal dominates). Above zero: clean energy.')
print('Watch the clean side swell during monsoon (Jul-Sep) as hydro and wind surge.')

---
## 7. Daily Generation Bar Race — Who Powers India Today?

An animated bar chart showing the day's generation by source, ranked. Watch coal always dominate, but the supporting cast changes with the seasons.

In [ ]:
# Bar chart race — daily fuel ranking
fuel_labels_map = dict(zip(fuel_cols, ['Coal', 'Lignite', 'Gas', 'Nuclear', 'Hydro', 'Wind', 'Solar', 'Other RE']))
fuel_color_map = {fuel_labels_map[k]: v for k, v in earth_sky.items()}

# Sample every 5 days for reasonable animation
sample_days = list(range(0, len(df), 5)) + [len(df)-1]

frames = []
for idx in sample_days:
    row = df.iloc[idx]
    vals = [(fuel_labels_map[f], row[f]) for f in fuel_cols]
    vals.sort(key=lambda x: x[1])  # ascending for horizontal bar
    names = [v[0] for v in vals]
    values = [v[1] for v in vals]
    colors = [fuel_color_map[n] for n in names]
    
    date_str = row['date'].strftime('%b %d')
    total = sum(values)
    
    frame = go.Frame(
        data=[go.Bar(
            x=values, y=names,
            orientation='h',
            marker_color=colors,
            text=[f'{v:.0f} MU' for v in values],
            textposition='outside',
            hovertemplate='%{y}: %{x:.0f} MU<extra></extra>',
        )],
        name=date_str,
        layout=dict(title=dict(
            text=f'Who Powers India? — {date_str} 2024<br><span style="font-size:14px">Total: {total:.0f} MU</span>',
        )),
    )
    frames.append(frame)

# Start with Jan 1
row0 = df.iloc[0]
vals0 = [(fuel_labels_map[f], row0[f]) for f in fuel_cols]
vals0.sort(key=lambda x: x[1])
names0 = [v[0] for v in vals0]
values0 = [v[1] for v in vals0]
colors0 = [fuel_color_map[n] for n in names0]

fig = go.Figure(
    data=[go.Bar(
        x=values0, y=names0,
        orientation='h',
        marker_color=colors0,
        text=[f'{v:.0f} MU' for v in values0],
        textposition='outside',
    )],
    frames=frames,
)

fig.update_layout(
    title=dict(text=f'Who Powers India? — Jan 01 2024<br><span style="font-size:14px">Total: {sum(values0):.0f} MU</span>',
              font=dict(size=18)),
    xaxis=dict(range=[0, df['coal'].max() * 1.15], title='Generation (MU/day)'),
    plot_bgcolor=bg, paper_bgcolor=bg,
    height=500, width=700,
    updatemenus=[dict(
        type='buttons', showactive=False,
        y=1.15, x=0.5, xanchor='center',
        buttons=[
            dict(label='▶ Play', method='animate',
                 args=[None, dict(frame=dict(duration=200, redraw=True),
                                  fromcurrent=True, mode='immediate',
                                  transition=dict(duration=150))]),
            dict(label='⏸ Pause', method='animate',
                 args=[[None], dict(frame=dict(duration=0, redraw=False),
                                    mode='immediate')]),
        ]
    )],
    sliders=[dict(
        active=0, pad=dict(t=60),
        steps=[dict(args=[[f.name], dict(frame=dict(duration=0, redraw=True), mode='immediate',
                                          transition=dict(duration=100))],
                    method='animate', label=f.name) for f in frames],
        currentvalue=dict(prefix='Date: ', font=dict(size=14)),
    )],
)

fig.show()
print('Coal is always #1, but watch the rank of Hydro, Wind, and Solar shift with seasons.')
print('During monsoon: Hydro and Wind climb up. In winter: Solar and Wind drop.')

---
## 8. Animated Radial Bloom — Building the Flower

The radial bloom from Notebook 10 — but animated. Watch each petal appear day by day, the flower growing through the year. Color = clean share (red→green).

In [ ]:
# Radial bloom animation using Plotly barpolar — no size limits
# Builds the flower in ~52 steps (weekly reveals)

n_days = len(df)
angles_deg = np.linspace(0, 360, n_days, endpoint=False)
width_deg = 360 / n_days * 0.95

max_total = df['total'].max()
heights = df['total'].values / max_total * 5

clean_pct = df['clean_share'].values
vmin, vmax = clean_pct.min(), clean_pct.max()

# Map clean_share to RdYlGn colors using plotly's sample_colorscale
from plotly.colors import sample_colorscale
norms = [(p - vmin) / (vmax - vmin) for p in clean_pct]
bar_colors = sample_colorscale('RdYlGn', norms)

# Build frames — reveal ~7 days at a time
frames = []
step = 7
frame_indices = list(range(step, n_days, step)) + [n_days - 1]

for idx in frame_indices:
    h = np.zeros(n_days)
    h[:idx+1] = heights[:idx+1]
    
    date_str = df.iloc[idx]['date'].strftime('%b %d')
    
    frame = go.Frame(
        data=[go.Barpolar(
            r=h,
            theta=angles_deg,
            width=[width_deg] * n_days,
            marker_color=bar_colors,
            marker_line_width=0,
            base=1.0,
            opacity=0.9,
        )],
        name=date_str,
    )
    frames.append(frame)

# Initial state — first week
h0 = np.zeros(n_days)
h0[:step+1] = heights[:step+1]

fig = go.Figure(
    data=[go.Barpolar(
        r=h0,
        theta=angles_deg,
        width=[width_deg] * n_days,
        marker_color=bar_colors,
        marker_line_width=0,
        base=1.0,
        opacity=0.9,
    )],
    frames=frames,
)

# Month label annotations around the circle
month_starts = [0, 31, 60, 91, 121, 152, 182, 213, 244, 274, 305, 335]
month_labels_short = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
annotations = []
for day, label in zip(month_starts, month_labels_short):
    angle_rad = (day / n_days) * 2 * np.pi
    x = 0.5 + 0.45 * np.sin(angle_rad)
    y = 0.5 + 0.45 * np.cos(angle_rad)
    annotations.append(dict(
        x=x, y=y, xref='paper', yref='paper',
        text=f'<b>{label}</b>', showarrow=False,
        font=dict(size=11, color='#666'),
    ))

fig.update_layout(
    title=dict(text='India Breathes — Radial Bloom Growing Day by Day', font=dict(size=18)),
    polar=dict(
        radialaxis=dict(visible=False, range=[0, 7]),
        angularaxis=dict(visible=False, direction='clockwise', rotation=90),
        bgcolor=bg,
    ),
    plot_bgcolor=bg, paper_bgcolor=bg,
    height=650, width=650,
    showlegend=False,
    annotations=annotations,
    updatemenus=[dict(
        type='buttons', showactive=False,
        y=1.1, x=0.5, xanchor='center',
        buttons=[
            dict(label='▶ Play', method='animate',
                 args=[None, dict(frame=dict(duration=100, redraw=True),
                                  fromcurrent=True, mode='immediate')]),
            dict(label='⏸ Pause', method='animate',
                 args=[[None], dict(frame=dict(duration=0, redraw=False),
                                    mode='immediate')]),
        ]
    )],
    sliders=[dict(
        active=0, pad=dict(t=50),
        steps=[dict(args=[[f.name], dict(frame=dict(duration=0, redraw=True), mode='immediate')],
                    method='animate', label=f.name) for f in frames],
        currentvalue=dict(prefix='Date: ', font=dict(size=14)),
    )],
)

fig.show()
print('Red petals = high coal share (dirty). Green petals = high clean share (monsoon).')
print('Watch the flower grow — winter starts red, monsoon glows green, winter returns.')

---
## 9. Wind+Solar vs Gas — The Displacement Story (Animated)

Watch the daily gap between wind+solar and gas grow through the year. Green bars = wind+solar wins. The gap is *massive*.

In [ ]:
# Animated gap chart: Wind+Solar vs Gas
frames = []
step = 5
frame_indices = list(range(step, len(df), step)) + [len(df)-1]

gap = df['wind_solar'] - df['gas']
gap_colors = ['#27AE60' if v > 0 else '#C0392B' for v in gap]

for idx in frame_indices:
    frame = go.Frame(
        data=[
            go.Scatter(x=df['date'].iloc[:idx+1], y=df['wind_solar'].iloc[:idx+1],
                      line=dict(width=2, color='#E67E22'), name='Wind+Solar',
                      fill='tozeroy', fillcolor='rgba(230,126,34,0.15)'),
            go.Scatter(x=df['date'].iloc[:idx+1], y=df['gas'].iloc[:idx+1],
                      line=dict(width=2, color=earth_sky['gas']), name='Gas',
                      fill='tozeroy', fillcolor='rgba(74,144,217,0.15)'),
        ],
        name=df.iloc[idx]['date'].strftime('%b %d'),
    )
    frames.append(frame)

d0 = df.iloc[:step+1]
fig = go.Figure(
    data=[
        go.Scatter(x=d0['date'], y=d0['wind_solar'],
                  line=dict(width=2, color='#E67E22'), name='Wind+Solar',
                  fill='tozeroy', fillcolor='rgba(230,126,34,0.15)'),
        go.Scatter(x=d0['date'], y=d0['gas'],
                  line=dict(width=2, color=earth_sky['gas']), name='Gas',
                  fill='tozeroy', fillcolor='rgba(74,144,217,0.15)'),
    ],
    frames=frames,
)

fig.update_layout(
    title=dict(text='Wind+Solar vs Gas — Daily Generation Race', font=dict(size=18)),
    xaxis=dict(range=[df['date'].min(), df['date'].max()]),
    yaxis=dict(range=[0, df['wind_solar'].max() * 1.1], title='Generation (MU/day)'),
    plot_bgcolor=bg, paper_bgcolor=bg,
    height=500,
    legend=dict(orientation='h', y=-0.1, x=0.5, xanchor='center'),
    updatemenus=[dict(
        type='buttons', showactive=False,
        y=1.15, x=0.5, xanchor='center',
        buttons=[
            dict(label='▶ Play', method='animate',
                 args=[None, dict(frame=dict(duration=60, redraw=True),
                                  fromcurrent=True, mode='immediate')]),
            dict(label='⏸ Pause', method='animate',
                 args=[[None], dict(frame=dict(duration=0, redraw=False),
                                    mode='immediate')]),
        ]
    )],
    sliders=[dict(
        active=0, pad=dict(t=50),
        steps=[dict(args=[[f.name], dict(frame=dict(duration=0, redraw=True), mode='immediate')],
                    method='animate', label=f.name) for f in frames],
        currentvalue=dict(prefix='Date: ', font=dict(size=14)),
    )],
)

fig.show()
ratio = df['wind_solar'].mean() / df['gas'].mean()
print(f'Wind+Solar is {ratio:.1f}x Gas on average — gas is becoming irrelevant.')
print(f'Wind+Solar won every single day of 2024. Not close.')

---
## 10. Clean Share Gauge — Month by Month

An animated gauge showing how clean the grid is, sweeping month by month. Watch it climb during monsoon and fall back in winter.

In [ ]:
# Monthly clean share for gauge
monthly_clean = df.groupby('month').agg(
    clean_share=('clean_share', 'mean'),
    total=('total', 'sum'),
    co2=('co2_total', lambda x: x.sum()/1e3),
).reset_index()

frames = []
for m in range(1, 13):
    row = monthly_clean[monthly_clean['month'] == m].iloc[0]
    frame = go.Frame(
        data=[go.Indicator(
            mode='gauge+number+delta',
            value=row['clean_share'],
            number=dict(suffix='%', font=dict(size=50)),
            delta=dict(reference=monthly_clean.loc[0, 'clean_share'] if m == 1 
                       else monthly_clean[monthly_clean['month'] == m-1].iloc[0]['clean_share'],
                       suffix='%'),
            gauge=dict(
                axis=dict(range=[10, 40], ticksuffix='%'),
                bar=dict(color='#2EC4B6'),
                bgcolor='#EEEEEE',
                steps=[
                    dict(range=[10, 20], color='#FADBD8'),
                    dict(range=[20, 30], color='#FDEBD0'),
                    dict(range=[30, 40], color='#D5F5E3'),
                ],
                threshold=dict(line=dict(color='#C0392B', width=3), value=23,
                              thickness=0.8),
            ),
            title=dict(text=f'Clean Energy Share — {month_names[m-1]} 2024<br>'
                           f'<span style="font-size:14px">CO2: {row["co2"]:.0f} Mt | Gen: {row["total"]/1e3:.0f} TWh</span>'),
        )],
        name=month_names[m-1],
    )
    frames.append(frame)

# Start with January
r0 = monthly_clean.iloc[0]
fig = go.Figure(
    data=[go.Indicator(
        mode='gauge+number',
        value=r0['clean_share'],
        number=dict(suffix='%', font=dict(size=50)),
        gauge=dict(
            axis=dict(range=[10, 40], ticksuffix='%'),
            bar=dict(color='#2EC4B6'),
            bgcolor='#EEEEEE',
            steps=[
                dict(range=[10, 20], color='#FADBD8'),
                dict(range=[20, 30], color='#FDEBD0'),
                dict(range=[30, 40], color='#D5F5E3'),
            ],
            threshold=dict(line=dict(color='#C0392B', width=3), value=23,
                          thickness=0.8),
        ),
        title=dict(text=f'Clean Energy Share — Jan 2024<br>'
                       f'<span style="font-size:14px">CO2: {r0["co2"]:.0f} Mt | Gen: {r0["total"]/1e3:.0f} TWh</span>'),
    )],
    frames=frames,
)

fig.update_layout(
    plot_bgcolor=bg, paper_bgcolor=bg,
    height=450, width=500,
    updatemenus=[dict(
        type='buttons', showactive=False,
        y=1.1, x=0.5, xanchor='center',
        buttons=[
            dict(label='▶ Play', method='animate',
                 args=[None, dict(frame=dict(duration=1000, redraw=True),
                                  fromcurrent=True, mode='immediate',
                                  transition=dict(duration=500))]),
            dict(label='⏸ Pause', method='animate',
                 args=[[None], dict(frame=dict(duration=0, redraw=False),
                                    mode='immediate')]),
        ]
    )],
    sliders=[dict(
        active=0, pad=dict(t=60),
        steps=[dict(args=[[f.name], dict(frame=dict(duration=0, redraw=True), mode='immediate',
                                          transition=dict(duration=300))],
                    method='animate', label=f.name) for f in frames],
        currentvalue=dict(prefix='Month: ', font=dict(size=14)),
    )],
)

fig.show()
peak = monthly_clean.loc[monthly_clean['clean_share'].idxmax()]
trough = monthly_clean.loc[monthly_clean['clean_share'].idxmin()]
print(f'Cleanest month: {month_names[int(peak["month"])-1]} ({peak["clean_share"]:.1f}%)')
print(f'Dirtiest month: {month_names[int(trough["month"])-1]} ({trough["clean_share"]:.1f}%)')
print(f'Red line = annual average ({df["clean_share"].mean():.1f}%)')

---
## What We've Got

| # | Animation | What It Shows | Feel |
|---|-----------|--------------|------|
| 1 | Racing Lines | W+S pulling away from Gas/Nuclear | Dominance |
| 2 | Monthly Donut | Fuel mix morphing by month | Seasonal shift |
| 3 | Weekly Heatmap | Generation painting itself week by week | Calendar rhythm |
| 4 | Emissions Heatmap | Dirty/clean days filling in | Pollution story |
| 5 | CO2 Race | Actual vs counterfactual diverging | What clean energy saves |
| 6 | Breathing Grid | Fossil down / clean up, drawing left to right | The breath |
| 7 | Bar Race | Daily fuel ranking changing with seasons | Who powers India |
| 8 | Radial Bloom | Flower growing day by day, red→green | The art piece |
| 9 | W+S vs Gas | The displacement story in motion | Irrelevance of gas |
| 10 | Clean Gauge | Monthly clean share gauge | The pulse |

All play in Jupyter. Hit ▶ on each one.